In [ ]:
# 라이브러리 임포트
import os  # 파일과 경로 처리
import urllib.request  # 파일 다운로드
import tarfile  # tar 파일 추출
import pickle  # 토크나이저 저장 및 로딩
import re  # 정규표현식
import time  # 시간 계산

def download_file(url, filename):
    """
    중복 다운로드를 막기 위해 파일 존재 여부를 확인하여 로컬에 없는 경우 URL에서 다운로드합니다.

    매개변수:
        url (str): 다운로드할 파일의 URL
        filename (str): 다운로드된 파일을 저장할 로컬 경로

    반환값:
        None: 다운로드 과정에 대한 상태 메시지를 출력합니다.
    """
    # 중복 다운로드를 막기 위해 파일이 존재하는지 확인합니다.
    if not os.path.exists(filename):
        print(f"{url}에서 파일을 다운로드합니다...")
        urllib.request.urlretrieve(url, filename)
        print("다운로드 완료.")
    else:
        print(f"{filename}이 이미 다운로드되어 있습니다.")

def is_within_directory(directory, target):
    """
    경로 탐색 공격(path traversal attack)을 막기 위해
    타깃 경로를 확인하는 보안 검사를 수행합니다.
    추출된 파일이 의도한 디렉토리 안에 있도록 보장합니다.

    매개변수:
        directory (str): 베이스 디렉토리 경로
        target (str): 타깃 경로

    반환값:
        bool: target이 directory 안에 있으면 True, 그렇지 않으면 False
    """
    # 비교를 위해 두 경로를 절대 경로로 바꿉니다.
    abs_directory = os.path.abspath(directory)
    abs_target = os.path.abspath(target)
    # 포함 관계를 확인하기 위해 공통 경로를 추출합니다.
    prefix = os.path.commonprefix([abs_directory, abs_target])
    return prefix == abs_directory

def safe_extract_tar(tar_file, required_files):
    """
    보안 검사를 통해 tar 파일을 안전하게 추출합니다.
    경로 탐색 공격을 방지하고 필요한 파일만 추출합니다.

    매개변수:
        tar_file (str): tar 압축 파일 경로
        required_files (list): 추출할 파일이름 리스트

    반환값:
        None: 파일을 추출하고 과정을 출력합니다.

    예외:
        Exception: 경로 탐색 공격이 감지되는 경우
    """
    with tarfile.open(tar_file, "r:gz") as tar:
        # 압축된 모든 파일에 대해 보안 검사를 수행합니다.
        for member in tar.getmembers():
            if not is_within_directory('.', member.name):
                raise Exception("Tar 파일에서 경로 탐색 공격이 감지되었습니다.")

        # 지정된 파일만 추출합니다.
        for member in tar.getmembers():
            if any(member.name.endswith(file) for file in required_files):
                # 안전을 위해 파일에서 경로를 삭제합니다.
                member.name = os.path.basename(member.name)
                tar.extract(member, '.')
                print(f"{member.name} 추출")

def create_word_generator(filepath):
    """
    텍스트 파일에서 한 번에 하나의 단어를 반환하는 제너레이터를 만듭니다.
    대규모 텍스트 파일을 메모리 효율적으로 처리하는 방법입니다.

    매개변수:
        filepath (str): 텍스트 파일 경로

    반환값:
        generator: 파일에 있는 개별 단어
    """
    def generator():
        with open(filepath, 'r') as f:
            for line in f:
                for word in line.split():
                    yield word
    return generator()

def download_and_prepare_data(url):
    """
    훈련 데이터셋을 다운로드, 추출, 준비합니다.
    다운로드와 보안 검사를 포함한 데이터 추출을 처리합니다.

    매개변수:
        url (str): 다운로드할 데이터셋 URL

    반환값:
        generator: 훈련 데이터를 위한 단어 제너레이터
    """
    required_files = ["train.txt", "test.txt"]
    filename = os.path.basename(url)

    # 필요한 경우 데이터셋을 다운로드합니다.
    download_file(url, filename)

    # 기존에 없는 경우 필요한 파일을 추출합니다.
    if not all(os.path.exists(file) for file in required_files):
        print("파일 추출...")
        safe_extract_tar(filename, required_files)
        print("추출 완료.")
    else:
        print("'train.txt'와 'test.txt'는 이미 추출되어 있습니다.")

    # 단어 제너레이터를 만들어 반환합니다.
    return create_word_generator("train.txt")

In [ ]:
from collections import defaultdict  # 토큰 및 쌍 카운트

#vocabulary 초기화
#단어를 문자의 sequence로 표현
def initialize_vocabulary(corpus):
  vocabulary=defaultdict(int)   #각 단어와 빈도수를 mapping한 dictionary
  charset=set()                 #corpus에 등장하는 고유한 모든 문자의 집합
  for word in corpus:
    #_를 각 문자의 시작 부분에 추가한다.
    #이를 통해 단어의 시작 부분에 등장하는 subword와 중간에 등장하는 subword을 구분한다.
    word_with_marker='_' + word
    #각 단어를 개별 문자로 분할한다.
    characters=list(word_with_marker)
    #단어에 등장하는 새로운 문자로 charset을 업데이트한다.
    #charset은 data structure가 set이므로 중복이 허용되지 않는다.
    charset.update(characters)
    #개별 문자를 공백으로 연결하여 해당 단어의 tokenization된 버전을 만든다.
    tokenized_word=' '.join(characters)
    vocabulary[tokenized_word] +=1
  return vocabulary, charset

In [ ]:
max_corpus_size = 500_000  # 처리할 최대 단어 개수
data_url = "https://www.thelmbook.com/data/news"  # 데이터셋

# 훈련 데이터를 다운로드하고 전처리합니다
word_gen = download_and_prepare_data(data_url)

# 말뭉치가 최대 크기에 도달할 때까지 단어를 모읍니다.
word_list = []
for word in word_gen:
    word_list.append(word)
    if len(word_list) >= max_corpus_size:
        break

https://www.thelmbook.com/data/news에서 파일을 다운로드합니다...
다운로드 완료.
파일 추출...


/tmp/ipykernel_3443/2938972042.py:74: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, '.')


train.txt 추출
test.txt 추출
추출 완료.


In [ ]:
dic=defaultdict(int)
print(f"{dic} -> type:{type(dic)}")

defaultdict(<class 'int'>, {}) -> type:<class 'collections.defaultdict'>


In [ ]:
corpus=word_list[:10]
charset=set()
for word in corpus:
  word_with_marker='_' + word
  characters=list(word_with_marker)
  print(f"{word} -> {list(word_with_marker)}")
  charset.update(characters)
  print(f"set -> {charset}")
  tokenized_word=' '.join(characters)
  dic[tokenized_word] +=1
dic

Tracey -> ['_', 'T', 'r', 'a', 'c', 'e', 'y']
set -> {'a', '_', 'e', 'T', 'r', 'c', 'y'}
Wygal -> ['_', 'W', 'y', 'g', 'a', 'l']
set -> {'a', '_', 'e', 'T', 'r', 'l', 'W', 'c', 'y', 'g'}
weighed -> ['_', 'w', 'e', 'i', 'g', 'h', 'e', 'd']
set -> {'h', 'a', 'i', '_', 'e', 'T', 'r', 'l', 'W', 'c', 'y', 'w', 'd', 'g'}
### -> ['_', '#', '#', '#']
set -> {'h', 'a', 'i', '_', 'e', 'T', 'r', 'l', 'W', 'c', 'y', 'w', 'd', 'g', '#'}
pounds -> ['_', 'p', 'o', 'u', 'n', 'd', 's']
set -> {'p', 'i', 'T', 'r', 'W', 'w', 'd', 'h', 's', '_', 'o', '#', 'a', 'e', 'l', 'n', 'y', 'g', 'c', 'u'}
before -> ['_', 'b', 'e', 'f', 'o', 'r', 'e']
set -> {'p', 'i', 'T', 'r', 'W', 'b', 'w', 'd', 'h', 's', '_', 'o', '#', 'a', 'e', 'l', 'n', 'y', 'g', 'f', 'c', 'u'}
starting -> ['_', 's', 't', 'a', 'r', 't', 'i', 'n', 'g']
set -> {'p', 'i', 'T', 'r', 'W', 'b', 'w', 'd', 'h', 's', '_', 'o', '#', 'a', 'e', 't', 'l', 'n', 'y', 'g', 'f', 'c', 'u'}
a -> ['_', 'a']
set -> {'p', 'i', 'T', 'r', 'W', 'b', 'w', 'd', 'h', 's',

defaultdict(int,
            {'_ T r a c e y': 1,
             '_ W y g a l': 1,
             '_ w e i g h e d': 1,
             '_ # # #': 1,
             '_ p o u n d s': 1,
             '_ b e f o r e': 1,
             '_ s t a r t i n g': 1,
             '_ a': 1,
             '_ ` `': 1,
             '_ c l e a n': 1})

In [ ]:
pair_counts=defaultdict(int)
for tokenized_word, count in dic.items():
  tokens=tokenized_word.split()
  for i in range(len(tokens)-1):
    pair=(tokens[i],tokens[i+1])
    pair_counts[pair] += count
pair_counts

defaultdict(int,
            {('_', 'T'): 1,
             ('T', 'r'): 1,
             ('r', 'a'): 1,
             ('a', 'c'): 1,
             ('c', 'e'): 1,
             ('e', 'y'): 1,
             ('_', 'W'): 1,
             ('W', 'y'): 1,
             ('y', 'g'): 1,
             ('g', 'a'): 1,
             ('a', 'l'): 1,
             ('_', 'w'): 1,
             ('w', 'e'): 1,
             ('e', 'i'): 1,
             ('i', 'g'): 1,
             ('g', 'h'): 1,
             ('h', 'e'): 1,
             ('e', 'd'): 1,
             ('_', '#'): 1,
             ('#', '#'): 2,
             ('_', 'p'): 1,
             ('p', 'o'): 1,
             ('o', 'u'): 1,
             ('u', 'n'): 1,
             ('n', 'd'): 1,
             ('d', 's'): 1,
             ('_', 'b'): 1,
             ('b', 'e'): 1,
             ('e', 'f'): 1,
             ('f', 'o'): 1,
             ('o', 'r'): 1,
             ('r', 'e'): 1,
             ('_', 's'): 1,
             ('s', 't'): 1,
             ('t', 'a'): 1,
   

In [ ]:
#초기화 후 BPE는 vocabulary에서 가장 빈도가 높은 token 쌍을 반복적으로 병합하며
#이런 쌍 사이에 있는 공백을 없애면서 점진적으로 긴 token을 형성한다.
def get_pair_counts(vocabulary):
  pair_counts=defaultdict(int)
  for tokenized_word, count in vocabulary.items():
    tokens = tokenized_word.split()     #vocabulary에 있는 tokenized_word를 token으로 분할한다.
    for i in range(len(tokens) - 1):
      pair = (tokens[i], tokens[i+1])   #인접한 token 쌍을 만든다.
      pair_counts[pair] += count        #단어 카운트를 기반으로 token 쌍의 카운트를 증가시킨다.

  return pair_counts         #token 쌍과 총 카운트의 dictionary

In [ ]:
most_frequent_pair=max(pair_counts,key=pair_counts.get)
print(most_frequent_pair,f"type:{type(most_frequent_pair)}")
new_vocabulary={}
bigram=re.escape(' '.join(most_frequent_pair))
print(bigram,f"type:{type(bigram)}")
pattern=re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
print(pattern)
for tokenized_word, count in dic.items():
  new_tokenized_word=pattern.sub("".join(most_frequent_pair),tokenized_word)
  new_vocabulary[new_tokenized_word]=count

new_vocabulary

('#', '#') type:<class 'tuple'>
\#\ \# type:<class 'str'>
re.compile('(?<!\\S)\\#\\ \\#(?!\\S)')


{'_ T r a c e y': 1,
 '_ W y g a l': 1,
 '_ w e i g h e d': 1,
 '_ ## #': 1,
 '_ p o u n d s': 1,
 '_ b e f o r e': 1,
 '_ s t a r t i n g': 1,
 '_ a': 1,
 '_ ` `': 1,
 '_ c l e a n': 1}

In [ ]:
#vocabulary의 모든 tokenization된 단어에 있는 입력 token 쌍을 병합한다.
#그런 다음 등장하는 모든 pair를 하나의 새로운 token으로 병합한 새로운 vocabulary을 반환한다.
def merge_pair(vocabulary, pair):
  new_vocabulary={}
  #문자열에 있는 특수문자에 자동으로 역슬래시 문자를 추가한다.
  bigram=re.escape(' '.join(pair))
  #매칭된 항목 앞과 뒤에 공백이 아닌 문자가 있는지 확인하여 bigram이 큰 단어의 일부에 매칭되지 않게 한다.
  pattern=re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
  for tokenized_word, count in vocabulary.items():
    #패턴에 매칭된 모든 bigram을 token쌍을 연결한 문자열로 교체하여 tokenization된 새 단어를 만든다.
    new_tokenized_word = pattern.sub("".join(pair), tokenized_word)
    new_vocabulary[new_tokenized_word] = count
  return new_vocabulary

In [ ]:
#가장 빈도가 높은 token 쌍을 반복적으로 병합하는 BPE algorithm을 구현한다.
def byte_pair_encoding(corpus,vocab_size):
  vocabulary, charset=initialize_vocabulary(corpus)
  merges=[]
  tokens=set(charset)
  #token개수가 vocab_size에 도달하거나 병합할 쌍이 남아 있지 않을 때까지 반복
  while len(tokens)<vocab_size:
    pair_counts=get_pair_counts(vocabulary)
    #남아 있는 쌍이 없는지 검사, 남아 있지 않으면 반복 종료
    if not pair_counts:
      break
    #가장 빈도가 높은 token 쌍을 찾는다.
    most_frequent_pair=max(pair_counts,key=pair_counts.get)
    merges.append(most_frequent_pair)
    #vocabulary을 통해 token 쌍이 병합된다.
    vocabulary=merge_pair(vocabulary,most_frequent_pair)
    #새로운 token을 만든다.
    new_token=''.join(most_frequent_pair)
    #새로운 token이 tokens 집합에 추가되며, 병합될 token쌍이 merges에 기록된다.
    tokens.add(new_token)
  return vocabulary,merges,charset,tokens

In [ ]:
v,m,c,t=byte_pair_encoding(word_list[:10], 100)

{'p', 'i', 'T', 'r', 'W', 'b', 'w', 'd', 'h', 's', '_', 'o', '#', 'a', 'e', 't', 'l', 'n', 'y', 'g', 'f', 'c', 'u', '`'} type:<class 'set'>
{'p', 'a', 'i', 'e', 't', 'T', 'r', 'l', 'W', 'b', 'n', 'y', 'w', 'd', 'g', 'h', 's', '_', 'o', 'f', 'c', 'u', '`', '#'} type:<class 'set'>
24
##
_T
_Tr
_Tra
_Trac
_Trace
_Tracey
_W
_Wy
_Wyg
_Wyga
_Wygal
_w
_we
_wei
_weig
_weigh
_weighe
_weighed
_##
_###
_p
_po
_pou
_poun
_pound
_pounds
_b
_be
_bef
_befo
_befor
_before
_s
_st
_sta
_star
_start
_starti
_startin
_starting
_a
_`
_``
_c
_cl
_cle
_clea
_clean


In [ ]:
#tokenizer 상태 저장 함수
def save_tokenizer(merges,charset,tokens,filename="tokenizer.pkl"):
  with open(filename,"wb") as f:
    pickle.dump({
        "merges":merges,
        "charset":charset,
        "tokens":tokens
    },f)

#tokenizer 상태 로드 함수
def load_tokenizer(filename="tokenizer.pkl"):
  with open(filename,"rb") as f:
    #tokenizer 구성요소가 담긴 dictionary
    return pickle.load(f)

In [ ]:
# 설정 파라미터
vocab_size = 5_000  # 어휘사전 크기
max_corpus_size = 500_000  # 처리할 최대 단어 개수
data_url = "https://www.thelmbook.com/data/news"  # 데이터셋

# 훈련 데이터를 다운로드하고 전처리합니다
word_gen = download_and_prepare_data(data_url)

# 말뭉치가 최대 크기에 도달할 때까지 단어를 모읍니다.
word_list = []
for word in word_gen:
    word_list.append(word)
    if len(word_list) >= max_corpus_size:
        break

# BPE 토크나이저를 훈련합니다.
print("BPE 토크나이저 훈련...")
vocab, merges, charset, tokens = byte_pair_encoding(word_list, vocab_size)

#훈련된 tokenizer 저장
#save_tokenizer(merges, charset, tokens)

news이 이미 다운로드되어 있습니다.
'train.txt'와 'test.txt'는 이미 추출되어 있습니다.
BPE 토크나이저 훈련...


In [ ]:
word_list[:10]

['Tracey',
 'Wygal',
 'weighed',
 '###',
 'pounds',
 'before',
 'starting',
 'a',
 '``',
 'clean']

In [ ]:
merges[:10]

[('_', 't'),
 ('h', 'e'),
 ('_', 'a'),
 ('i', 'n'),
 ('r', 'e'),
 ('_', '.'),
 ('_', 's'),
 ('_t', 'he'),
 ('o', 'n'),
 ('e', 'r')]

In [ ]:
list(charset)[:20]

['E',
 'Y',
 'w',
 'h',
 'G',
 'ω',
 'â',
 '/',
 'I',
 'S',
 'R',
 '€',
 '\u200f',
 'e',
 'l',
 'Z',
 'C',
 '“',
 'Â',
 '©']

In [ ]:
list(tokens)[:10]

['_Brazil',
 '_caught',
 'ober',
 '_z',
 '_lives',
 '_Com',
 '_sen',
 'rel',
 'ook',
 '_ran']

In [ ]:
len(tokens)

5000

#훈련된 BPE tokenizer test

In [ ]:
#훈련된 tokenizer로 단어를 tokenization한다.
def tokenize_word(word, merges, vocabulary,charset,unk_token="<UNK>"):
  word = '_' + word
  if word in vocabulary:
    return [word]
  tokens=[char if char in charset else unk_token for char in word]
  #1
  for left, right in merges:
    i=0
    while i<len(tokens)-1:
      if tokens[i:i+2] == [left,right]:
        tokens[i:i+2] = [left+right]
      else:
        i+=1
  return tokens

In [ ]:
#token 쌍을 병합 결과에 매핑하는 dictionary를 만든다.
#일관된 tokenization를 위해 병합 순서를 유지한다.
def build_merge_map(merges):
  merge_map={}
  #병합 우선순위로 매핑을 만든다.
  for i, (left, right) in enumerate(merges):
    merged_token = left + right
    merge_map[(left, right)] = (merged_token, i)
  return merge_map

#미리 계산된 병합 매핑을 사용해 tokenization 함수를 최적화한다.
#원본 알고리즘과 동일하지만 더 빠르게 결과를 생성한다.
def tokenize_word_fast(word, merge_map, vocabulary, charset, unk_token="<UNK>"):
  #단어가 vocabulary에 있는지 확인한다.
  word = '_' + word
  if word in vocabulary:
    return [word]

  #단어를 문자로 분할하고 알지 못하는 문자를 대체한다.
  tokens = [char if char in charset else unk_token for char in word]

  #더 이상 가능한 병합이 없을 때까지 병합을 계속한다.
  while True:
    #가능한 모든 병합 연산을 찾는다.
    pairs_with_positions=[]
    for i in range(len(tokens) - 1):
      pair=(tokens[i],tokens[i+1])
      if pair in merge_map:
        merged_token, merge_priority=merge_map[pair]
        pairs_with_positions.append((i,pair,merged_token,merge_priority))
    #더 이상 가능한 병합이 없으면 종료한다.
    if not pairs_with_positions:
      break

    #일관성을 위해 병합 우선순위와 위치로 정렬한다.
    pairs_with_positions.sort(key=lambda x: (x[3], x[0]))
    #유효한 첫 번째 병합을 적용한다.
    pos, pair, merged_token, _=pairs_with_positions[0]
    tokens[pos:pos+2] = [merged_token]
  return tokens

In [ ]:
#tokenizer loading
#tokenizer = load_tokenizer()

# 로딩된 토크나이저로 샘플 문장을 토큰화합니다.
sentence = "Let's proceed to the language modeling part."

start_time = time.time()
tokenized_sentence = [tokenize_word(word, merges, vocab, charset) for word in sentence.split()]
elapsed = time.time() - start_time
print("\n일반적인 구현으로 토큰화한 문장:")
for word, tokens in zip(sentence.split(), tokenized_sentence):
    print(f"{word} -> {tokens}")
print("--- 소요시간: %s 초 ---" % (elapsed))

In [ ]:
merge_map=build_merge_map(merges)
start_time=time.time()
fast_tokenized_sentence = [tokenize_word_fast(word,merge_map,vocab,charset) for word in sentence.split()]
elapsed=time.time()-start_time
print("\n빠른 구현으로 토큰화한 문장:")
for word, tokens in zip(sentence.split(), fast_tokenized_sentence):
  print(f"{word} -> {tokens}")
print("--- 소요시간: %s 초 ---" % (time.time() - start_time))